# 10 · End-to-end pipeline: fail safely, then publish clean input

Execute the same pipeline twice under the same strict policy: the dirty batch must FAIL; the clean batch must PASS. All artifacts are run-scoped. Silver is a candidate dataset until a publication manifest approves Gold. This is an educational single-writer pipeline, not a transactional table system.


## Environment
Upload the prepared sample files before the session, then use notebook 00 to check the configured storage. This notebook then runs independently, top to bottom. Spark 3.5 is the target; no Hive catalog is used. Set `BASE_PATH` in the following cell or set `DQ_BASE_PATH` in the driver environment.


In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("OrderDataQuality").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", "4")  # tiny teaching datasets only
spark.conf.set("spark.sql.ansi.enabled", "true")
spark.conf.set("spark.sql.csv.parser.columnPruning.enabled", "false")
BASE_PATH = os.environ.get(
    "DQ_BASE_PATH", "s3://YOUR-BUCKET/training/order-quality"
).rstrip("/")
# On EMR/Glue edit the default above if driver environment variables are unavailable.
# Spark VM: hdfs:///user/student/order-quality ; local: file:///tmp/order-quality
assert (
    "YOUR-BUCKET" not in BASE_PATH
), "Set DQ_BASE_PATH or edit BASE_PATH before running"
PROCESSING_DATE = os.environ.get("DQ_PROCESSING_DATE", "2024-01-03")
RUN_ID = uuid.uuid4().hex
RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUARANTINE_PATH, AUDIT_PATH = [
    f"{BASE_PATH}/{layer}"
    for layer in ["raw", "bronze", "silver", "gold", "quarantine", "audit"]
]
print("Spark", spark.version, "storage", BASE_PATH, "run", RUN_ID)


## Shared readers, transformations and policy
These are the same explicit schemas and policies used in earlier lessons. Input envelopes remain the unit of reconciliation.


In [ ]:
ORDER_FIELDS = [
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "unit_price",
    "discount",
    "status",
    "order_date",
    "shipped_date",
    "cancellation_reason",
    "order_total",
    "seller_id",
    "country",
    "arrival_date",
]
order_schema = T.StructType(
    [T.StructField(c, T.StringType(), True) for c in ORDER_FIELDS]
    + [T.StructField("_corrupt_record", T.StringType(), True)]
)


def read_orders(path):
    # Preserve one source envelope per physical JSON line, including malformed JSON.
    # Source row IDs are materialized before branching; raw_text supports replay.
    raw = (
        spark.read.text(path)
        .withColumnRenamed("value", "raw_text")
        .withColumn("source_file", F.input_file_name())
        .withColumn("source_row_id", F.monotonically_increasing_id())
    )
    parsed = raw.withColumn(
        "parsed",
        F.from_json(
            "raw_text",
            order_schema,
            {"mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt_record"},
        ),
    )
    return parsed.select("source_row_id", "source_file", "raw_text", "parsed.*").cache()


orders = read_orders(f"{RAW_PATH}/orders")
orders.count()  # materialize once before splitting
customers = spark.read.option("header", True).csv(f"{RAW_PATH}/customers")
products = spark.read.option("header", True).csv(f"{RAW_PATH}/products")
items = spark.read.option("header", True).csv(f"{RAW_PATH}/order_items")
orders.show(30, truncate=False)

POLICY = dict(
    max_reject_percentage=1.0,
    max_null_customer_percentage=0.5,
    max_duplicate_orders=0,
    require_reconciliation=True,
)


def decide(metrics, policy=POLICY):
    failures = []
    if metrics["rows"] == 0:
        failures.append("empty input")
    if metrics["reject_pct"] > policy["max_reject_percentage"]:
        failures.append("reject percentage")
    if metrics["null_customer_pct"] > policy["max_null_customer_percentage"]:
        failures.append("customer completeness")
    if metrics["duplicate_orders"] > policy["max_duplicate_orders"]:
        failures.append("duplicate orders")
    if policy["require_reconciliation"] and not metrics["reconciled"]:
        failures.append("reconciliation")
    if failures:
        return "FAIL", failures
    near_limit = metrics["reject_pct"] > 0.8 * policy["max_reject_percentage"]
    return ("WARN" if near_limit else "PASS"), []


## Run a complete pipeline
The schema gate checks producer keys before projection and blocks unexpected or missing required fields. Malformed envelopes are recorded by ingestion checks and quarantined; an explicit 10% parser budget controls whether Bronze is allowed. Missing/null values in present fields are row checks. Audit is written before a failed gate returns. The final gate also includes item-total consistency.


In [ ]:
import json


def run_pipeline(order_path, item_path, label):
    run = uuid.uuid4().hex
    global orders, items, typed, scored, valid, rejected, RUN_ID
    RUN_ID = run
    orders = read_orders(order_path)
    source_count = orders.count()
    # For this tiny lesson collect raw JSON objects to inspect producer key contracts.
    # Large workloads should use a distributed producer-schema contract implementation.
    schema_problems = []
    malformed = 0
    for row in orders.select("raw_text").collect():
        try:
            obj = json.loads(row.raw_text)
        except json.JSONDecodeError:
            malformed += 1
            continue
        if not isinstance(obj, dict):
            schema_problems.append("JSON root must be an object")
            continue
        missing = sorted(set(ORDER_FIELDS) - set(obj))
        extra = sorted(set(obj) - set(ORDER_FIELDS))
        wrong_types = [
            k
            for k, v in obj.items()
            if k in ORDER_FIELDS and v is not None and not isinstance(v, str)
        ]
        if missing or extra or wrong_types:
            schema_problems.append(str((missing, extra, wrong_types)))
    ingestion_pass = (
        source_count > 0 and 100.0 * malformed / max(source_count, 1) <= 10.0
    )
    schema_pass = not schema_problems
    early_path = f"{AUDIT_PATH}/pipeline/{run}/ingestion"
    spark.createDataFrame(
        [
            (
                run,
                label,
                source_count,
                malformed,
                ingestion_pass,
                schema_pass,
                str(schema_problems),
            )
        ],
        "run_id string, scenario string, source_rows long, malformed_rows long, ingestion_pass boolean, schema_pass boolean, schema_problems string",
    ).write.mode("errorifexists").parquet(early_path)
    if not ingestion_pass or not schema_pass:
        orders.write.mode("errorifexists").parquet(
            f"{QUARANTINE_PATH}/pipeline/{run}/blocked_input"
        )
        print(label, "FAIL before Bronze; inspect", early_path)
        return {"run_id": run, "status": "FAIL", "gold_path": None}
    orders.write.mode("errorifexists").parquet(f"{BRONZE_PATH}/pipeline/{run}")
    # Re-read persisted Bronze to stabilize source occurrence IDs across branches.
    orders.unpersist()
    orders = spark.read.parquet(f"{BRONZE_PATH}/pipeline/{run}").cache()
    orders.count()
    items = spark.read.option("header", True).csv(item_path)
    typed = (
        orders.withColumn("customer_id_clean", F.trim("customer_id"))
        .withColumn("status_clean", F.upper(F.trim("status")))
        .withColumn("qty", F.expr("try_cast(quantity as int)"))
        .withColumn("price", F.expr("try_cast(unit_price as decimal(18,2))"))
        .withColumn("discount_value", F.expr("try_cast(discount as decimal(8,2))"))
        .withColumn("total", F.expr("try_cast(order_total as decimal(18,2))"))
        .withColumn("event_date", F.expr("try_cast(order_date as date)"))
        .withColumn("shipped_on", F.expr("try_cast(shipped_date as date)"))
        .withColumn("arrived_on", F.expr("try_cast(arrival_date as date)"))
    )
    # Reference tables are deduplicated for membership joins, not as a silent repair.
    # A separate customer-key check still exposes duplicate reference records.
    customer_keys = (
        customers.select(F.col("customer_id").alias("customer_id_clean"))
        .distinct()
        .withColumn("known_customer", F.lit(True))
    )
    product_keys = (
        products.select("product_id")
        .distinct()
        .withColumn("known_product", F.lit(True))
    )
    typed = (
        typed.join(customer_keys, "customer_id_clean", "left")
        .join(product_keys, "product_id", "left")
        .withColumn("key_count", F.count("*").over(Window.partitionBy("order_id")))
    )
    # A predicate means PASS. NULL is a failure unless the rule explicitly permits it.
    rules = [
        (
            "DQ000",
            "parseable",
            "raw_text",
            "Malformed JSON",
            F.col("_corrupt_record").isNull() & F.col("order_id").isNotNull(),
        ),
        (
            "DQ001",
            "customer present",
            "customer_id",
            "Missing or blank customer",
            F.length("customer_id_clean") > 0,
        ),
        (
            "DQ002",
            "positive integer quantity",
            "quantity",
            "Not an integer in 1..1000",
            F.col("qty").between(1, 1000),
        ),
        (
            "DQ003",
            "nonnegative price",
            "unit_price",
            "Invalid or negative price",
            F.col("price") >= 0,
        ),
        (
            "DQ004",
            "allowed status",
            "status",
            "Unknown status",
            F.col("status_clean").isin("CREATED", "PAID", "SHIPPED", "CANCELLED"),
        ),
        (
            "DQ005",
            "unique order key",
            "order_id",
            "Duplicate business key; quarantine all copies",
            F.col("key_count") == 1,
        ),
        (
            "DQ006",
            "event window",
            "order_date",
            "Invalid, future, old or outside daily window",
            F.col("event_date") == F.date_sub(F.to_date(F.lit(PROCESSING_DATE)), 1),
        ),
        (
            "DQ007",
            "known customer",
            "customer_id",
            "Customer reference not found",
            F.coalesce(F.col("known_customer"), F.lit(False)),
        ),
        (
            "DQ008",
            "known product",
            "product_id",
            "Product reference not found",
            F.coalesce(F.col("known_product"), F.lit(False)),
        ),
        (
            "DQ009",
            "discount range",
            "discount",
            "Discount outside 0..100",
            F.col("discount_value").between(0, 100),
        ),
        (
            "BR001",
            "shipment date",
            "shipped_date",
            "SHIPPED requires valid shipped_date",
            (F.col("status_clean") != "SHIPPED") | F.col("shipped_on").isNotNull(),
        ),
        (
            "BR002",
            "cancellation reason",
            "cancellation_reason",
            "CANCELLED requires reason",
            (F.col("status_clean") != "CANCELLED")
            | (F.length(F.trim("cancellation_reason")) > 0),
        ),
        (
            "BR003",
            "nonnegative total",
            "order_total",
            "Invalid or negative order total",
            F.col("total") >= 0,
        ),
        (
            "BR004",
            "order arithmetic",
            "order_total",
            "Header differs from quantity times unit price",
            F.abs(F.col("total") - F.col("qty") * F.col("price"))
            <= F.lit("0.01").cast("decimal(18,2)"),
        ),
    ]
    failure_structs = [
        F.when(
            ~F.coalesce(predicate, F.lit(False)),
            F.struct(
                F.lit(rule_id).alias("dq_rule_id"),
                F.lit(name).alias("dq_rule_name"),
                F.lit(column).alias("dq_column"),
                F.lit(reason).alias("dq_reason"),
            ),
        )
        for rule_id, name, column, reason, predicate in rules
    ]
    scored = (
        typed.withColumn(
            "dq_failures", F.filter(F.array(*failure_structs), lambda x: x.isNotNull())
        )
        .withColumn(
            "dq_status",
            F.when(F.size("dq_failures") == 0, "VALID").otherwise("INVALID"),
        )
        .withColumn("pipeline_run_id", F.lit(RUN_ID))
        .withColumn("processing_timestamp", F.current_timestamp())
        .cache()
    )
    scored.count()
    valid = scored.filter("dq_status = 'VALID'")
    rejected = scored.filter("dq_status = 'INVALID'")
    valid.select("order_id", "quantity", "status", "dq_status").show(truncate=False)
    rejected.select("order_id", "source_row_id", "dq_failures").show(30, truncate=False)
    item_values = items.withColumn("q", F.expr("try_cast(quantity as int)")).withColumn(
        "p", F.expr("try_cast(unit_price as decimal(18,2))")
    )
    line_totals = item_values.groupBy("order_id").agg(
        F.sum(F.col("q") * F.col("p")).alias("item_total"),
        F.sum(
            (~F.coalesce((F.col("q") > 0) & (F.col("p") >= 0), F.lit(False))).cast(
                "int"
            )
        ).alias("invalid_items"),
    )
    scored = scored.join(line_totals, "order_id", "left")
    item_pass = F.coalesce(
        (F.col("invalid_items") == 0)
        & (
            F.abs(F.col("total") - F.col("item_total"))
            <= F.lit("0.01").cast("decimal(18,2)")
        ),
        F.lit(False),
    )
    additional_failure = F.struct(
        F.lit("BR005").alias("dq_rule_id"),
        F.lit("item total agreement").alias("dq_rule_name"),
        F.lit("order_total").alias("dq_column"),
        F.lit("Missing/invalid items or header mismatch").alias("dq_reason"),
    )
    scored = scored.withColumn(
        "dq_failures",
        F.when(item_pass, F.col("dq_failures")).otherwise(
            F.concat("dq_failures", F.array(additional_failure))
        ),
    )
    scored = scored.withColumn(
        "dq_status", F.when(F.size("dq_failures") == 0, "VALID").otherwise("INVALID")
    ).cache()
    scored.count()
    valid = scored.filter("dq_status='VALID'")
    rejected = scored.filter("dq_status='INVALID'")
    valid.write.mode("errorifexists").parquet(f"{SILVER_PATH}/pipeline/{run}")
    rejected.write.mode("errorifexists").parquet(
        f"{QUARANTINE_PATH}/pipeline/{run}/records"
    )
    # Reconcile persisted outputs, not only the in-memory split.
    accepted_disk = spark.read.parquet(f"{SILVER_PATH}/pipeline/{run}")
    rejected_disk = spark.read.parquet(f"{QUARANTINE_PATH}/pipeline/{run}/records")
    accounted = accepted_disk.unionByName(rejected_disk)

    def total(df, col):
        return df.agg(F.coalesce(F.sum(col), F.lit(0)).alias("v")).first().v

    rec_rows = []
    rec_values = [
        ("rows", source_count, accounted.count()),
        ("quantity", total(typed, "qty"), total(accounted, "qty")),
        ("amount", total(typed, "total"), total(accounted, "total")),
        (
            "uncastable quantity",
            typed.filter("qty is null").count(),
            accounted.filter("qty is null").count(),
        ),
        (
            "uncastable amount",
            typed.filter("total is null").count(),
            accounted.filter("total is null").count(),
        ),
    ]
    for name, source, target in rec_values:
        rec_rows.append(
            (
                run,
                name,
                str(source),
                str(target),
                str(source - target),
                "0",
                "PASS" if source == target else "FAIL",
            )
        )
    rec = spark.createDataFrame(
        rec_rows,
        "run_id string, check_name string, source_value string, target_value string, difference string, tolerance string, status string",
    )
    rec.write.mode("errorifexists").parquet(
        f"{AUDIT_PATH}/pipeline/{run}/reconciliation"
    )
    identity_ok = (
        orders.select("source_row_id")
        .exceptAll(accounted.select("source_row_id"))
        .count()
        == 0
        and accounted.select("source_row_id")
        .exceptAll(orders.select("source_row_id"))
        .count()
        == 0
    )
    rejected_count = rejected_disk.count()
    null_count = typed.filter(
        "customer_id_clean is null or customer_id_clean = ''"
    ).count()
    duplicates = typed.filter("key_count > 1").select("order_id").distinct().count()
    orphan_items = items.join(
        orders.select("order_id").distinct(), "order_id", "left_anti"
    ).count()
    metrics = dict(
        rows=source_count,
        reject_pct=100.0 * rejected_count / source_count,
        null_customer_pct=100.0 * null_count / source_count,
        duplicate_orders=duplicates,
        reconciled=identity_ok
        and all(s == t for _, s, t in rec_values)
        and orphan_items == 0,
    )
    status, reasons = decide(metrics)
    # Rule audit is persisted for FAIL as well as PASS, with zero-failure rules included.
    failures = scored.select(
        "source_row_id", F.explode("dq_failures").alias("failure")
    ).select("source_row_id", "failure.*")
    failures.groupBy("dq_rule_id", "dq_rule_name").count().show(truncate=False)
    audit_rows = []
    for rid, rname in [(r[0], r[1]) for r in rules] + [
        ("BR005", "item total agreement")
    ]:
        failed = (
            failures.filter(F.col("dq_rule_id") == rid)
            .select("source_row_id")
            .distinct()
            .count()
        )
        audit_rows.append(
            (
                run,
                "orders",
                rid,
                rname,
                "failure_percentage",
                100.0 * failed / source_count,
                0.0,
                failed,
                source_count,
                100.0 * failed / source_count,
                "PASS" if failed == 0 else "FAIL",
            )
        )
    audit = spark.createDataFrame(
        audit_rows,
        "run_id string, dataset string, rule_id string, rule_name string, metric_name string, metric_value double, threshold double, failed_records long, total_records long, failure_percentage double, status string",
    ).withColumn("processing_timestamp", F.current_timestamp())
    audit.write.mode("errorifexists").parquet(f"{AUDIT_PATH}/dq_results/{run}")
    spark.createDataFrame(
        [(run, label, status, json.dumps(metrics), json.dumps(reasons))],
        "run_id string, scenario string, status string, metrics string, reasons string",
    ).withColumn("processing_timestamp", F.current_timestamp()).write.mode(
        "errorifexists"
    ).parquet(
        f"{AUDIT_PATH}/pipeline/{run}/gate"
    )
    gold_path = None
    if status in ("PASS", "WARN"):
        gold_path = f"{GOLD_PATH}/pipeline/{run}"
        accepted_disk.groupBy("order_date", "seller_id", "country").agg(
            F.sum("total").alias("sales"), F.count("*").alias("orders")
        ).write.mode("errorifexists").parquet(gold_path)
        # Written last: consumers use only runs with a successful publication manifest.
        spark.createDataFrame(
            [(run, gold_path, status)], "run_id string, gold_path string, status string"
        ).write.mode("errorifexists").json(f"{AUDIT_PATH}/published/{run}")
    print(label, status, metrics, reasons)
    return dict(run_id=run, status=status, gold_path=gold_path)


## Expected failure, then successful publication
Do not assert that the dirty source has zero valid rows: O016 or other individual rows can change as contracts evolve. Assert that publication is blocked. The clean fixture uses a new key and matching items to demonstrate the same gate passing.


In [ ]:
dirty_result = run_pipeline(
    f"{RAW_PATH}/orders", f"{RAW_PATH}/order_items", "intentionally dirty"
)
assert dirty_result["status"] == "FAIL" and dirty_result["gold_path"] is None
clean_result = run_pipeline(
    f"{RAW_PATH}/clean", f"{RAW_PATH}/clean_items", "clean control"
)
assert clean_result["status"] == "PASS"
spark.read.parquet(clean_result["gold_path"]).show(truncate=False)
assert spark.read.parquet(clean_result["gold_path"]).first()["sales"] == 50
for result in [dirty_result, clean_result]:
    run = result["run_id"]
    print("Inspect run", run)
    for layer in [BRONZE_PATH, SILVER_PATH]:
        spark.read.parquet(f"{layer}/pipeline/{run}").show(3, truncate=False)
    spark.read.parquet(f"{QUARANTINE_PATH}/pipeline/{run}/records").select(
        "order_id", "dq_failures"
    ).show(30, truncate=False)
    spark.read.parquet(f"{AUDIT_PATH}/pipeline/{run}/gate").show(truncate=False)


## Final exercise
Repair the producer data without dropping envelopes, regenerate fixtures, and rerun. Explain every change in accepted counts and financial totals. Add a new rule with a stable ID, then show its audit record. For production: persist failure audit around write exceptions too, use orchestrator idempotency keys, restrict consumers to manifests, and choose a transactional table format if atomic multi-table publication is required.
